# Week 5 — GroupBy & Aggregation: Time Series & Multi-Key GroupBy
## Phase 2a Python | PORA Academy Cohort 7

By the end of this session, you will be able to:
- Parse datetime columns with `pd.to_datetime()`
- Extract year, month, day of week from datetime
- GroupBy with datetime components
- GroupBy with two keys (cross-tabulation)

---

## 2. Why This Session Matters

On Wednesday you learned to split a dataset into groups and aggregate each group. Today we point that same skill at **time**.

Olist's order history spans roughly two years — from late 2016 through October 2018. Once you can work with dates, you can finally answer the questions a business actually cares about:

- **Is business growing?** Compare orders in 2016, 2017, and 2018.
- **When do people shop most?** Break a year down month by month.
- **Which state is growing fastest?** Group by state *and* year at the same time.

Here's a concrete preview. If you count Olist's 2017 orders month by month, almost every month sits between 800 and 5,700 orders — except **November 2017, which explodes to 7,544 orders**. That spike is **Black Friday**, the single biggest shopping event of the Brazilian retail calendar. You cannot see that spike without time-series groupby. By the end of today, you'll surface it in two lines of code.

Time turns a flat table into a story. Let's learn to read it.

## 3. Setup

We mount Google Drive and load the two tables we need today: `orders` (one row per order, with timestamps) and `customers` (which tells us each order's state).

In [ ]:
import pandas as pd
from google.colab import drive
import os

drive.mount('/content/drive')
olist_path = '/content/drive/MyDrive/olist-data'

orders = pd.read_csv(os.path.join(olist_path, 'olist_orders_dataset.csv'))
customers = pd.read_csv(os.path.join(olist_path, 'olist_customers_dataset.csv'))

print(f"✓ orders: {len(orders):,} rows")       # Expected: 99,441
print(f"✓ customers: {len(customers):,} rows")  # Expected: 99,441

## 4. Concept 1: Datetime Parsing

When pandas reads a CSV, it has no way of knowing that `order_purchase_timestamp` is a *date*. It sees text like `"2017-11-24 19:30:00"` and stores it as a plain **string** (dtype `object`).

Strings are useless for time analysis. You can't ask "what year is this?" or "which day of the week?" from a string. The fix is one function: **`pd.to_datetime()`**. It converts the text column into a real `datetime64` type, and that unlocks the `.dt` accessor — a toolbox for pulling out year, month, day name, and more.

**Step 1 — check the dtype, then convert it:**

In [ ]:
print(orders['order_purchase_timestamp'].dtype)   # Expected: object

orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

print(orders['order_purchase_timestamp'].dtype)   # Expected: datetime64[ns]

**Step 2 — extract components with the `.dt` accessor.**

Once a column is `datetime64`, `.dt` lets you pull out any piece of the date. We create four new columns we'll reuse all session: numeric year, numeric month, readable month name, and the day-of-week name.

In [ ]:
orders['year'] = orders['order_purchase_timestamp'].dt.year
orders['month'] = orders['order_purchase_timestamp'].dt.month
orders['month_name'] = orders['order_purchase_timestamp'].dt.month_name()
orders['day_of_week'] = orders['order_purchase_timestamp'].dt.day_name()

orders[['order_purchase_timestamp', 'year', 'month', 'month_name', 'day_of_week']].head()

**Step 3 — orders by year.** Now that `year` is a real column, grouping by it answers "is business growing?" in one line.

In [ ]:
print(orders.groupby('year')['order_id'].count())
# Expected:
# year
# 2016      329
# 2017    45101
# 2018    54011

The growth story is clear: 2016 was a tiny launch year (only 329 orders), then 2017 and 2018 are both full-scale operations. (2018 ends in October — more on that in the Mini Challenge.)

**Step 4 — zoom into a single year, month by month.** To see *seasonality*, filter to one complete year and group by month.

In [ ]:
orders_2017 = orders[orders['year'] == 2017]
monthly_2017 = orders_2017.groupby('month')['order_id'].count()
print(monthly_2017)
# Expected:
# month
# 1      800
# 2     1780
# 3     2682
# 4     2404
# 5     3700
# 6     3245
# 7     4026
# 8     4331
# 9     4285
# 10    4631
# 11    7544   <- Black Friday peak!
# 12    5673

There it is — **November 2017 = 7,544 orders**, nearly 3,000 more than any other month. That's the Black Friday spike we promised. Time-series groupby made it visible instantly.

### 🔍 Going Deeper: readable labels & plotting

Numeric months (`1`, `2`, `3`...) are fine for sorting but hard to read in a chart. Grouping by `month_name` instead gives human-friendly labels like `January`, `February`. The trade-off: month names sort *alphabetically*, not chronologically, so `April` comes before `January`. Use numeric `month` when order matters, `month_name` when readability matters.

And because `monthly_2017` is a Series indexed by month, you can plot it directly with `monthly_2017.plot(kind='bar')` — the Black Friday spike jumps off the page. We'll do real charting in a later week; for now, just know the data is one step away from a graph.

### ⚠️ Common Mistakes

The most common datetime mistake is trying to slice the **string** to grab the year *before* parsing. It sometimes works, but it's fragile and breaks the moment a date is formatted differently. Always parse first, then use `.dt`.

In [ ]:
# ❌ Wrong — trying to groupby the original string column before parsing
# orders.groupby(orders['order_purchase_timestamp'].str[:4])['order_id'].count()
# This is messy and error-prone

# ✅ Correct — parse to datetime first, then use .dt.year
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['year'] = orders['order_purchase_timestamp'].dt.year
print(orders.groupby('year')['order_id'].count())
# Expected:
# year
# 2016      329
# 2017    45101
# 2018    54011

## 5. Concept 2: Multi-Key GroupBy

So far we've grouped by *one* key at a time — year, or month. But the best questions need **two** keys at once: "orders per state **per year**", "sales per category **per month**". You pass a **list** of keys to `groupby()`.

The catch: a two-key groupby returns a result with a *multi-level index* that's awkward to read as a long list. The fix is **`.unstack()`** — it pivots one of the keys out into columns, turning a stacked list into a clean cross-tabulation (rows × columns), exactly like a pivot table in Excel.

**Orders per state per year.** The state lives in the `customers` table, so we merge first, then group by `['customer_state', 'year']`.

In [ ]:
state_year = orders.merge(
    customers, on='customer_id'
).groupby(['customer_state', 'year'])['order_id'].count().unstack(fill_value=0)

print(state_year.loc[['SP', 'RJ', 'MG']])
# Each row = a state, each column = a year (2016, 2017, 2018).
# SP totals across all years = 41,746 | RJ = 12,852 | MG = 11,635
print("\nSP total across years:", state_year.loc['SP'].sum())  # Expected: 41746
print("RJ total across years:", state_year.loc['RJ'].sum())    # Expected: 12852
print("MG total across years:", state_year.loc['MG'].sum())    # Expected: 11635

Read it like a grid: each row is a state, each column is a year, each cell is the order count for that state-and-year. São Paulo (SP) dominates every year — it's Olist's home market.

**Day-of-week orders.** A single-key groupby on the `day_of_week` column we built earlier tells us *which day Brazilians shop*.

In [ ]:
dow_counts = orders.groupby('day_of_week')['order_id'].count()
print(dow_counts.sort_values(ascending=False))

# Sanity check: every order falls on exactly one weekday, so the total must be 99,441
print("\nTotal:", dow_counts.sum())   # Expected: 99441

## ⏱ Mini Challenge (5 minutes)

The dataset ends on **October 17, 2018** — that's why **Sep 2018 has only 16 orders** and **Oct 2018 has only 4**. Those months are *truncated*, not a real drop in business.

- Filter to 2018 data and print monthly order counts.
- Which months look complete? Which are clearly truncated?

In [ ]:
# Your turn: filter orders to year == 2018, then groupby month and count order_id.
orders_2018 = orders[orders['year'] == 2018]
monthly_2018 = orders_2018.groupby('year')['order_id'].count()
print(monthly_2018)

# Expected tail of the result:
# 9     16   <- truncated (only part of September)
# 10     4   <- truncated (data stops Oct 17)
# Months 1-8 are complete; months 9 and 10 are cut off.

## 7. Group Exercise (40 min)

Work in your group. **Use DeepSeek to help if you get stuck — but validate every output** against the expected values in the comments. AI writes code fast; *you* are responsible for checking it's correct.

Complete the four tasks below.

**Task 1 — Verify the year distribution.** Group orders by `year` and confirm the counts match.

In [ ]:
# TODO: groupby 'year' and count order_id
yearly = orders.groupby('year')['order_id'].count()
print(yearly)
# Expected: 2016=329, 2017=45101, 2018=54011

**Task 2 — Monthly count for 2017, find the peak.** Filter to 2017, group by month, and identify the busiest month.

In [ ]:
# TODO: filter to 2017, groupby 'month', count, then find the peak month with .idxmax()
orders_2017 = orders[orders['year'] == 2017]
monthly_2017 = orders_2017.groupby('month')['order_id'].count()
print(monthly_2017)
print("\nPeak month:", monthly_2017.idxmax(), "with", monthly_2017.max(), "orders")
# Expected: Peak month = 11 (November) with 7544 orders

**Task 3 — Average order price per month for 2017.** The price lives in the `items` table. Load it, merge with 2017 orders, then group by month and average `price`.

In [ ]:
# TODO: load order items, merge with orders_2017, groupby month, mean of 'price'
items = pd.read_csv(os.path.join(olist_path, 'olist_order_items_dataset.csv'))

orders_2017_items = orders_2017.merge(items, on='order_id')
avg_price_by_month = orders_2017_items.groupby('month')['price'].mean()
print(avg_price_by_month.round(2))
# Hint: there's no single 'right' number to memorize here — sanity-check that you get
# 12 rows (one per month) and prices land in a believable range (roughly 100-160 BRL).

**Task 4 — DeepSeek task.** Ask DeepSeek: *"Write pandas code showing the number of orders per `day_of_week` from a DataFrame called `orders` that has a `day_of_week` column."* Run its answer below, then **validate the total sums to 99,441**.

In [ ]:
# TODO: paste DeepSeek's code here, then validate the total
dow = orders.groupby('day_of_week')['order_id'].count()
print(dow.sort_values(ascending=False))

# assert dow.sum() == 99441  # validated against real dataset
print("\n✅ Validated: total =", dow.sum())   # Expected: 99441

## 8. Summary Table & Coming Up Next Session

Today you turned a flat table into a time-aware analysis.

| Skill | What it does | Key code |
|---|---|---|
| Datetime parsing | Convert string dates to real dates | `pd.to_datetime(col)` |
| Extract components | Pull year / month / day name | `.dt.year`, `.dt.month`, `.dt.day_name()` |
| Time-series groupby | Count orders by year or month | `groupby('year')['order_id'].count()` |
| Multi-key groupby | Group by two keys at once | `groupby(['state', 'year'])` |
| Unstack | Pivot a key into columns (cross-tab) | `.unstack(fill_value=0)` |

**The headline finding:** November 2017 = 7,544 orders — the Black Friday spike, visible only once you can group by time.

**Next session — Week 6: Data Cleaning.** Real datasets are messy: missing values, duplicates, inconsistent text. You'll learn to detect and fix these problems so your aggregations stay trustworthy. See you Wednesday!